## Importing libraries and data

In [7]:
pip install TensorFlow

   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/331.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/331.9 MB 1.7 MB/s eta 0:03:18
   ---------------------------------------- 0.8/331.9 MB 1.1 MB/s eta 0:05:06
   ---------------------------------------- 1.0/331.9 MB 1.1 MB/s eta 0:05:09
   ---------------------------------------- 1.3/331.9 MB 1.2 MB/s eta 0:04:46
   ---------------------------------------- 1.3/331.9 MB 1.2 MB/s eta 0:04:46
   ---------------------------------------- 1.6/331.9 MB 1.0 MB/s eta 0:05:16
   ---------------------------------------- 2.1/331.9 MB 1.2 MB/s eta 0:04:39
   ---------------------------------------- 2.6/331.9 MB 1.3 MB/s eta 0:04:16
   ---------------------------------------- 3.1/331.9 MB 1.4 MB/s eta 0:03:50
   ---------------------------------------- 3.7/331.9 MB 1.5 MB/s eta 0:03:37
   ----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from numpy import unique
from numpy import reshape
from keras.models import Sequential
from keras.layers import Conv1D, Conv2D, Dense, BatchNormalization, Flatten, MaxPooling1D, Dropout, LSTM
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [3]:
#Create a path to where your data is stored.
path = r'/Users/april/Machine Learning'

In [4]:
#Create a path to where your data is stored.
path2 = r'/Users/april/Machine Learning/Datasets'

In [5]:
# Import unscaled weather data
df_unscaled = pd.read_csv(os.path.join(path, 'Dataset-weather-prediction-dataset-processed.csv'))


In [6]:
#Read in the pleasant weather data.
pleasantweather = pd.read_csv(os.path.join(path2, 'Supervised','Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'))
pleasantweather

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22945,20221027,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22946,20221028,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22947,20221029,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22948,20221030,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [7]:
# Drop all Gdansk, Roma,Tours columns from df_unscaled since they are not included in pleasant weather data
# We know this from previous exercises using the same data
cols_to_drop = [col for col in df_unscaled.columns if col.startswith(('GDANSK', 'ROMA', 'TOURS'))]
df_unscaled = df_unscaled.drop(columns=cols_to_drop)


In [8]:
# Trying to find out all the different measurement types for each location

# Extract location names 
locations = set([col.split('_')[0] for col in df_unscaled.columns])

# Create a dictionary to store measurement counts for each location
measurement_counts = {location: {} for location in locations}

# Count occurrences of each measurement type for each location
for col in df_unscaled.columns:
    parts = col.split('_') 
    location = parts[0] 
    measurement = '_'.join(parts[1:])  # Join remaining parts if there are more than two

    if measurement not in measurement_counts[location]:
        measurement_counts[location][measurement] = 1
    else:
        measurement_counts[location][measurement] += 1

# Print the measurement counts for each location
for location, measurements in measurement_counts.items():
    print(f"Location: {location}")
    for measurement, count in measurements.items():
        print(f"  - {measurement}: {count}")
    print()

Location: MONTH
  - : 1

Location: MADRID
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: DEBILT
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: BUDAPEST
  - cloud_cover: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: LJUBLJANA
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1

Location: MAASTRICHT
  - cloud_cover: 1
  - wind_speed: 1
  - humidity: 1
  - pressure: 1
  - global_radiation: 1
  - precipitation: 1
  - sunshine: 1
  - temp_mean: 1
  - temp_min: 1
  - temp_max: 1


In [9]:
# Columns to drop
cols_to_drop = [col for col in df_unscaled.columns if col.endswith(('wind_speed', 'snow_depth'))]

# Dropping
df_unscaled = df_unscaled.drop(columns=cols_to_drop)

In [10]:
# Define relationships between locations
location_pairs = {
    'KASSEL': 'LJUBLJANA',
    'STOCKHOLM': 'OSLO',
    'MUNCHENB': 'SONNBLICK'
}

# Define the desired order of measurements
measurement_order = ['cloud_cover', 'humidity', 'pressure', 'global_radiation', 
                     'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max']

# Function to fill missing values and insert in correct position
def fill_missing_values(df_unscaled, location, measurement, neighbor):
    """
    Fills missing values for a given location and measurement using data from a neighbor location.
    Inserts the new column in the correct position based on the measurement order.

    Args:
        df_unscaled: The DataFrame containing the weather data.
        location: The location with missing values.
        measurement: The measurement with missing values.
        neighbor: The neighboring location to use for filling.

    Returns:
        The updated DataFrame with filled missing values and columns in the correct order.
    """
    source_col = f'{neighbor}_{measurement}'
    target_col = f'{location}_{measurement}'

    # Determine the insertion index 
    if measurement == measurement_order[0]:  # If it's the first measurement for the location
        # Find the index of the first column for the location (or 0 if no location columns exist)
        location_columns = [col for col in df_unscaled.columns if col.startswith(location)]
        if location_columns:
            insert_index = df_unscaled.columns.get_loc(location_columns[0]) 
        else:
            insert_index = 0
    else:
        insert_index = df_unscaled.columns.get_loc(f'{location}_{measurement_order[measurement_order.index(measurement) - 1]}') + 1 

    # Create the new column with missing values and insert it at the correct position
    df_unscaled.insert(insert_index, target_col, np.nan) 

    # Fill missing values in the new column
    df_unscaled[target_col].fillna(df_unscaled[source_col], inplace=True) 

    return df_unscaled

# Fill missing values for each location and measurement
for location, neighbor in location_pairs.items():
    for measurement in measurement_order:
        if f'{location}_{measurement}' not in df_unscaled.columns:  # Check if column already exists
            df_unscaled = fill_missing_values(df_unscaled, location, measurement, neighbor)

# Checking new columns for existance and location
selected_columns = [col for col in df_unscaled.columns if col.startswith(('KASSEL', 'STOCKHOLM', 'MUNCHENB'))]
print(df_unscaled[selected_columns])

       KASSEL_cloud_cover  KASSEL_humidity  KASSEL_pressure  \
0                1.205492         0.449867        -0.801741   
1                0.371461         0.818506        -0.897454   
2                1.205492         1.279304        -0.382997   
3                0.371461         0.910666         1.543227   
4                0.788477         0.818506         1.208231   
...                   ...              ...              ...   
22945           -0.462569        -0.010931        -0.000145   
22946           -0.879584        -0.010931        -0.000145   
22947           -0.879584        -0.010931        -0.000145   
22948           -0.879584        -0.010931        -0.000145   
22949           -0.879584        -0.010931        -0.000145   

       KASSEL_global_radiation  KASSEL_precipitation  KASSEL_sunshine  \
0                    -1.069690              0.747355        -0.647708   
1                    -1.267817              0.199693        -1.074723   
2                    -1.

C:\Users\april\AppData\Local\Temp\ipykernel_18984\2400176209.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_unscaled[target_col].fillna(df_unscaled[source_col], inplace=True)
C:\Users\april\AppData\Local\Temp\ipykernel_18984\2400176209.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha

In [11]:
# Checking new columns for existance and location
selected_columns = [col for col in df_unscaled.columns if col.startswith(('MUNCHENB'))]
print(df_unscaled[selected_columns])

       MUNCHENB_cloud_cover  MUNCHENB_humidity  MUNCHENB_pressure  \
0                 -0.093824          -0.746542           0.095234   
1                  0.318678          -0.344471           0.067319   
2                  0.318678           1.183399           0.132454   
3                  0.318678           1.102985           0.418586   
4                 -0.093824           0.700914           0.388345   
...                     ...                ...                ...   
22945             -1.331330          -0.022814          -0.000143   
22946              0.318678          -0.505299          -0.000143   
22947              0.731180          -0.987784          -0.000143   
22948              0.318678          -0.103228          -0.000143   
22949             -0.093824           0.540086          -0.000143   

       MUNCHENB_global_radiation  MUNCHENB_precipitation  MUNCHENB_sunshine  \
0                      -1.244144               -0.282933          -1.098059   
1            

In [12]:
# Export cleaned weather data
df_unscaled.to_csv(os.path.join(path, 'weather_clean.csv'), index=False)


In [25]:
# Rename df's
X = df_unscaled
y = pleasantweather

In [26]:
# Convert df's to arrays
X = np.array(X)
y = np.array(y)


In [27]:
# Reshaping X as a 3D object
X = X.reshape(-1,15,9)


In [28]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)


In [29]:
epochs = 20
batch_size = 16
n_hidden = 16

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [30]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 16)             │         1,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15)             │           255 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,919 (7.50 KB)

 Trainable params: 1,919 (7.50 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [32]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.0637 - loss: 10.4687 - val_accuracy: 0.0417 - val_loss: 9.3492
Epoch 2/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.2352 - loss: 10.7017 - val_accuracy: 0.2684 - val_loss: 9.4791
Epoch 3/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.2782 - loss: 10.5788 - val_accuracy: 0.2722 - val_loss: 9.4288
Epoch 4/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.2857 - loss: 10.5233 - val_accuracy: 0.2717 - val_loss: 9.5301
Epoch 5/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.2925 - loss: 10.5007 - val_accuracy: 0.2740 - val_loss: 9.5944
Epoch 6/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.3360 - loss: 10.5293 - val_accuracy: 0.2724 - val_loss: 9.7048
Epoch 7/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4202 - loss: 10.5132 - val_accuracy: 0.5444 - val_loss: 9.8327
Epoch 8/20
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4673 - loss: 10

In [33]:
# Define list of stations names
stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'
}

In [34]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [35]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Pred        BASEL  MADRID
True                     
BASEL        3681       1
BELGRADE     1092       0
BUDAPEST      213       1
DEBILT         82       0
DUSSELDORF     29       0
HEATHROW       82       0
KASSEL         11       0
LJUBLJANA      61       0
MAASTRICHT      9       0
MADRID        455       3
MUNCHENB        8       0
OSLO            5       0
STOCKHOLM       4       0
VALENTIA        1       0


In [36]:
epochs = 30
batch_size = 16
n_hidden = 32

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 32)             │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,871 (22.93 KB)

 Trainable params: 5,871 (22.93 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [39]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.1913 - loss: 11.3606 - val_accuracy: 0.2567 - val_loss: 9.9108
Epoch 2/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.2673 - loss: 11.6222 - val_accuracy: 0.2590 - val_loss: 9.9838
Epoch 3/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.2693 - loss: 11.4609 - val_accuracy: 0.2539 - val_loss: 10.0674
Epoch 4/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3808 - loss: 11.5352 - val_accuracy: 0.6406 - val_loss: 10.2270
Epoch 5/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5616 - loss: 11.6284 - val_accuracy: 0.6417 - val_loss: 10.3645
Epoch 6/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6269 - loss: 11.7417 - val_accuracy: 0.6413 - val_loss: 10.6579
Epoch 7/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6400 - loss: 11.9279 - val_accuracy: 0.6417 - val_loss: 10.8992
Epoch 8/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6430 - los

In [40]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [41]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Pred        BASEL  MADRID
True                     
BASEL        3680       2
BELGRADE     1092       0
BUDAPEST      213       1
DEBILT         82       0
DUSSELDORF     29       0
HEATHROW       82       0
KASSEL         11       0
LJUBLJANA      61       0
MAASTRICHT      9       0
MADRID        458       0
MUNCHENB        8       0
OSLO            5       0
STOCKHOLM       4       0
VALENTIA        1       0


In [42]:
epochs = 30
batch_size = 16
n_hidden = 64

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [43]:
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,919 (77.81 KB)

 Trainable params: 19,919 (77.81 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [45]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)

Epoch 1/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.0514 - loss: 25.0738 - val_accuracy: 0.0188 - val_loss: 24.2254
Epoch 2/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0792 - loss: 25.3878 - val_accuracy: 0.1225 - val_loss: 28.6420
Epoch 3/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0706 - loss: 25.0093 - val_accuracy: 0.0329 - val_loss: 23.4943
Epoch 4/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0608 - loss: 25.0858 - val_accuracy: 0.0493 - val_loss: 22.7467
Epoch 5/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0583 - loss: 24.9759 - val_accuracy: 0.0840 - val_loss: 25.9175
Epoch 6/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.1073 - loss: 24.9212 - val_accuracy: 0.0544 - val_loss: 25.4643
Epoch 7/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.0934 - loss: 24.8404 - val_accuracy: 0.1014 - val_loss: 25.8597
Epoch 8/30
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.0887 - l

In [46]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [47]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  KASSEL  LJUBLJANA  \
True                                                                           
BASEL           0       518      1758       0         310     187        416   
BELGRADE        1       173       240       5          49      34        283   
BUDAPEST        0        21        48       0          20      14         58   
DEBILT          0        11         6       0           5       6         29   
DUSSELDORF      0         3         2       0           2       0         14   
HEATHROW        0         8        14       1          12       6         28   
KASSEL          0         3         1       0           1       0          0   
LJUBLJANA       0         8        18       2           7       1         15   
MAASTRICHT      0         0         3       0           4       0          1   
MADRID          0        34       216       2          70      17         61   

In [48]:
epochs = 25
batch_size = 16
n_hidden = 8

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(MaxPooling1D())
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='tanh'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [49]:
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 14, 8)          │           152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 7, 8)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 8)              │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 15)             │           135 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 831 (3.25 KB)

 Trainable params: 831 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])


In [51]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)


Epoch 1/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.1282 - loss: 20.0023 - val_accuracy: 0.1863 - val_loss: 17.2315
Epoch 2/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1045 - loss: 18.5563 - val_accuracy: 0.1311 - val_loss: 17.3131
Epoch 3/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0604 - loss: 25.4486 - val_accuracy: 5.2283e-04 - val_loss: 23.4296
Epoch 4/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1290 - loss: 25.0983 - val_accuracy: 0.0995 - val_loss: 23.4713
Epoch 5/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1101 - loss: 24.8385 - val_accuracy: 6.9711e-04 - val_loss: 27.5718
Epoch 6/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0771 - loss: 24.2532 - val_accuracy: 0.0012 - val_loss: 25.3082
Epoch 7/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.0838 - loss: 23.8069 - val_accuracy: 0.0491 - val_loss: 25.4567
Epoch 8/25
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.

In [52]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [53]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Pred        BELGRADE  BUDAPEST  MADRID  SONNBLICK
True                                             
BASEL           2253      1244     151         34
BELGRADE         803       182     107          0
BUDAPEST         166        19      29          0
DEBILT            81         0       1          0
DUSSELDORF        27         1       1          0
HEATHROW          64         8      10          0
KASSEL            11         0       0          0
LJUBLJANA         35        15      11          0
MAASTRICHT         5         3       1          0
MADRID           175       207      75          1
MUNCHENB           4         3       1          0
OSLO               4         1       0          0
STOCKHOLM          4         0       0          0
VALENTIA           1         0       0          0


In [54]:
epochs = 10
batch_size = 4
n_hidden = 4

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(LSTM(n_hidden, input_shape=(timesteps, input_dim)))
model.add(Dropout(0.5))
model.add(Dense(n_classes, activation='sigmoid'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [55]:
model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 4)              │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 15)             │            75 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 299 (1.17 KB)

 Trainable params: 299 (1.17 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
model.compile(loss='categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [57]:
model.fit(X_train,
          y_train,
          batch_size=batch_size,
          validation_data=(X_test, y_test),
          epochs=epochs)


Epoch 1/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.0744 - loss: 9.2779 - val_accuracy: 0.0638 - val_loss: 9.1056
Epoch 2/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.1892 - loss: 10.0398 - val_accuracy: 0.3050 - val_loss: 9.8027
Epoch 3/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.2661 - loss: 10.4928 - val_accuracy: 0.3059 - val_loss: 10.2976
Epoch 4/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.2778 - loss: 10.9882 - val_accuracy: 0.3057 - val_loss: 10.9529
Epoch 5/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.2772 - loss: 11.6265 - val_accuracy: 0.3041 - val_loss: 11.5420
Epoch 6/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.2776 - loss: 12.2456 - val_accuracy: 0.3039 - val_loss: 12.0019
Epoch 7/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.3324 - loss: 12.7597 - val_accuracy: 0.3041 - val_loss: 12.5499
Epoch 8/10
4303/4303 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.374

In [58]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [59]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Pred        BASEL  MADRID
True                     
BASEL        1487    2195
BELGRADE      990     102
BUDAPEST      205       9
DEBILT         82       0
DUSSELDORF     28       1
HEATHROW       72      10
KASSEL         11       0
LJUBLJANA      48      13
MAASTRICHT      5       4
MADRID        198     260
MUNCHENB        7       1
OSLO            4       1
STOCKHOLM       4       0
VALENTIA        0       1


## CNN Model

In [60]:
epochs = 10
batch_size = 4
n_hidden = 4

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [61]:
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 14, 4)          │            76 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 14, 16)         │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,851 (7.23 KB)

 Trainable params: 1,851 (7.23 KB)

 Non-trainable params: 0 (0.00 B)

In [62]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [63]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)


Epoch 1/10
4303/4303 - 9s - 2ms/step - accuracy: 0.1524 - loss: 6009.6436
Epoch 2/10
4303/4303 - 7s - 2ms/step - accuracy: 0.1621 - loss: 55619.4336
Epoch 3/10
4303/4303 - 8s - 2ms/step - accuracy: 0.1637 - loss: 187416.8906
Epoch 4/10
4303/4303 - 7s - 2ms/step - accuracy: 0.1635 - loss: 428543.4688
Epoch 5/10
4303/4303 - 8s - 2ms/step - accuracy: 0.1644 - loss: 804471.5625
Epoch 6/10
4303/4303 - 7s - 2ms/step - accuracy: 0.1635 - loss: 1335748.7500
Epoch 7/10
4303/4303 - 8s - 2ms/step - accuracy: 0.1675 - loss: 2052045.1250
Epoch 8/10
4303/4303 - 8s - 2ms/step - accuracy: 0.1648 - loss: 2989983.5000
Epoch 9/10
4303/4303 - 8s - 2ms/step - accuracy: 0.1664 - loss: 4198057.0000
Epoch 10/10
4303/4303 - 7s - 2ms/step - accuracy: 0.1637 - loss: 5629592.5000


In [64]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [65]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL         247       207       820      99          74       149     155   
BELGRADE        0        55       657       3          17        26       0   
BUDAPEST        1         6        63       0           9        18       0   
DEBILT          0         0        10       0           7        24       0   
DUSSELDORF      0         0         2       2           1         4       0   
HEATHROW        0         1         4       1           3        29       0   
KASSEL          0         0         2       0           1         1       0   
LJUBLJANA       1         0         9       0           1         1       0   
MAASTRICHT      0         0         1       1           2         0       0   
MADRID          8        22        48       6           4        49       1   
MUNCHENB   

## CNN Model Retest

In [66]:
epochs = 15
batch_size = 8
n_hidden = 8

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax'))


C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [67]:
model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_2 (Conv1D)               │ (None, 14, 8)          │           152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 14, 16)         │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,991 (7.78 KB)

 Trainable params: 1,991 (7.78 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [69]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)


Epoch 1/15
2152/2152 - 5s - 2ms/step - accuracy: 0.1378 - loss: 2897.4573
Epoch 2/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1530 - loss: 26223.1797
Epoch 3/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1634 - loss: 87352.4453
Epoch 4/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1636 - loss: 190248.1719
Epoch 5/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1630 - loss: 344765.0000
Epoch 6/15
2152/2152 - 5s - 2ms/step - accuracy: 0.1614 - loss: 557048.2500
Epoch 7/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1597 - loss: 834707.4375
Epoch 8/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1604 - loss: 1192452.5000
Epoch 9/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1613 - loss: 1601409.8750
Epoch 10/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1622 - loss: 2122229.2500
Epoch 11/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1624 - loss: 2730732.5000
Epoch 12/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1641 - loss: 3461343.2500
Epoch 13/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1647 - loss: 4319233.0000
Epoch 

In [70]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [71]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Pred        BASEL  BELGRADE  BUDAPEST  DEBILT  DUSSELDORF  HEATHROW  KASSEL  \
True                                                                          
BASEL          77       608       363     220         253       204      77   
BELGRADE        0       682        57      10          40        21       0   
BUDAPEST        0        61         9       9          14        14       0   
DEBILT          0         5         4      14          11        10       0   
DUSSELDORF      0         2         1       2           2         3       0   
HEATHROW        0         7         0       6           6        16       0   
KASSEL          0         3         0       1           0         1       0   
LJUBLJANA       1        12         0       1           0         1       0   
MAASTRICHT      0         1         0       2           1         0       0   
MADRID          1        51        11      15          10        45       0   
MUNCHENB   

## CNN Final Test

In [72]:
epochs = 15
batch_size = 8
n_hidden = 8

timesteps = len(X_train[0])
input_dim = len(X_train[0][0])
n_classes = len(y_train[0])

model = Sequential()
model.add(Conv1D(n_hidden, kernel_size=2, activation='relu', input_shape=(timesteps, input_dim)))
model.add(Dense(16, activation='relu'))
model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='tanh'))

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [73]:
model.summary()


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 14, 8)          │           152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 14, 16)         │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 7, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 112)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 15)             │         1,695 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,991 (7.78 KB)

 Trainable params: 1,991 (7.78 KB)

 Non-trainable params: 0 (0.00 B)

In [74]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [75]:
model.fit(X_train, y_train, batch_size=batch_size, epochs=epochs, verbose=2)


Epoch 1/15
2152/2152 - 5s - 2ms/step - accuracy: 0.1421 - loss: 25.4925
Epoch 2/15
2152/2152 - 4s - 2ms/step - accuracy: 0.1017 - loss: 25.1936
Epoch 3/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0469 - loss: 25.9856
Epoch 4/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0457 - loss: 25.9054
Epoch 5/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0715 - loss: 25.5315
Epoch 6/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0714 - loss: 24.4441
Epoch 7/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0296 - loss: 25.0127
Epoch 8/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0073 - loss: 24.7511
Epoch 9/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0152 - loss: 25.0273
Epoch 10/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0204 - loss: 25.4925
Epoch 11/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0232 - loss: 25.3351
Epoch 12/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0272 - loss: 24.8719
Epoch 13/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0213 - loss: 25.3399
Epoch 14/15
2152/2152 - 4s - 2ms/step - accuracy: 0.0420 - l

In [76]:
def confusion_matrix(y_true, y_pred):
    y_true = pd.Series([stations[y] for y in np.argmax(y_true, axis=1)])
    y_pred = pd.Series([stations[y] for y in np.argmax(y_pred, axis=1)])

    return pd.crosstab(y_true, y_pred, rownames=['True'], colnames=['Pred'])


In [77]:
# Evaluate
print(confusion_matrix(y_test, model.predict(X_test)))


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Pred        BASEL  BELGRADE  KASSEL  MAASTRICHT  OSLO  STOCKHOLM
True                                                            
BASEL           1      3574       2          31    38         36
BELGRADE        0      1062       0           7    14          9
BUDAPEST        1       206       0           0     4          3
DEBILT          0        75       0           3     2          2
DUSSELDORF      0        27       0           1     1          0
HEATHROW        0        80       0           1     0          1
KASSEL          0        10       0           0     0          1
LJUBLJANA       0        61       0           0     0          0
MAASTRICHT      0         9       0           0     0          0
MADRID          0       456       0           2     0          0
MUNCHENB        0         8       0           0     0          0
OSLO            0         4       0           0     1          0
STOCKHOLM       0         4       0           0  